In [3]:
import pickle
from pathlib import Path
import numpy as np
import pandas as pd

# Change this line:
# PROJECT_ROOT = Path(__file__).resolve().parent.parent

# To this in Jupyter Notebook:
PROJECT_ROOT = Path.cwd().parent  # Adjust .parent steps based on where your notebook is saved
DATA_DIR = PROJECT_ROOT / "data"
SPLIT_DIR = DATA_DIR / "splits"

def verify_pipeline():
    # 1. Load files
    master = pd.read_csv(DATA_DIR / "materials_master.csv")
    tabular = pd.read_csv(DATA_DIR / "tabular_features.csv")
    train_ids = pd.read_csv(SPLIT_DIR / "train_ids.csv")["material_id"].tolist()
    val_ids = pd.read_csv(SPLIT_DIR / "val_ids.csv")["material_id"].tolist()
    test_ids = pd.read_csv(SPLIT_DIR / "test_ids.csv")["material_id"].tolist()

    print("--- 1. DATASET ALIGNMENT & INTEGRITY ---")
    print(f"Master materials count:  {len(master)}")
    print(f"Tabular features count: {len(tabular)}")
    print(f"Split counts -> Train: {len(train_ids)} | Val: {len(val_ids)} | Test: {len(test_ids)}")
    
    total_split_count = len(train_ids) + len(val_ids) + len(test_ids)
    assert total_split_count == len(master), f"Mismatch: Split total ({total_split_count}) != Master count ({len(master)})"
    print("✓ All 10,000 samples accounted for across splits.")

    # 2. Check Material ID Overlap Leakage
    set_train, set_val, set_test = set(train_ids), set(val_ids), set(test_ids)
    id_leak = (set_train & set_val) | (set_train & set_test) | (set_val & set_test)
    assert len(id_leak) == 0, f"ID Leakage detected: {id_leak}"
    print("✓ Zero material_id overlap between splits.")

    # 3. Check Formula Leakage (Polymorphs)
    train_formulas = set(master[master["material_id"].isin(set_train)]["formula"])
    val_formulas = set(master[master["material_id"].isin(set_val)]["formula"])
    test_formulas = set(master[master["material_id"].isin(set_test)]["formula"])

    formula_leak = (train_formulas & val_formulas) | (train_formulas & test_formulas) | (val_formulas & test_formulas)
    print(f"✓ Formula Leakage Check: {len(formula_leak)} overlapping chemical formulas.")
    if len(formula_leak) > 0:
        print(f"  WARNING: Leaked formulas: {list(formula_leak)[:5]}")

    # 4. Target Distribution Analysis (Bias Check)
    print("\n--- 2. TARGET DISTRIBUTION (log_bulk_modulus_vrh) ---")
    df_train = master[master["material_id"].isin(set_train)]
    df_val = master[master["material_id"].isin(set_val)]
    df_test = master[master["material_id"].isin(set_test)]

    target_col = "log_bulk_modulus_vrh" if "log_bulk_modulus_vrh" in master.columns else "bulk_modulus_vrh"

    stats = []
    for name, df in [("Train", df_train), ("Val", df_val), ("Test", df_test)]:
        vals = df[target_col].values
        stats.append({
            "Split": name,
            "Count": len(vals),
            "Mean": np.mean(vals),
            "Std": np.std(vals),
            "Min": np.min(vals),
            "Median": np.median(vals),
            "Max": np.max(vals),
        })

    df_stats = pd.DataFrame(stats)
    print(df_stats.to_string(index=False))

    # 5. Check missing values in tabular features
    nan_count = tabular.isna().sum().sum()
    print(f"\n--- 3. FEATURE COMPLETENESS ---")
    print(f"Total NaNs in tabular features: {nan_count}")
    if nan_count == 0:
        print("✓ Tabular feature matrix is clean.")

if __name__ == "__main__":
    verify_pipeline()

--- 1. DATASET ALIGNMENT & INTEGRITY ---
Master materials count:  10000
Tabular features count: 10000
Split counts -> Train: 7021 | Val: 1491 | Test: 1488
✓ All 10,000 samples accounted for across splits.
✓ Zero material_id overlap between splits.
✓ Formula Leakage Check: 0 overlapping chemical formulas.

--- 2. TARGET DISTRIBUTION (log_bulk_modulus_vrh) ---
Split  Count     Mean      Std       Min   Median      Max
Train   7021 1.885025 0.380239 -1.008774 1.929950 2.691606
  Val   1491 1.879221 0.400685 -0.283162 1.928437 2.584909
 Test   1488 1.889082 0.380321 -0.254145 1.946668 2.613860

--- 3. FEATURE COMPLETENESS ---
Total NaNs in tabular features: 9


In [6]:
import pickle
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent  # Adjust .parent steps based on where your notebook is saved
DATA_DIR = PROJECT_ROOT / "data"
SPLIT_DIR = DATA_DIR / "splits"

tabular = pd.read_csv(DATA_DIR / "tabular_features.csv")

with open(DATA_DIR / "impute_stats.pkl", "rb") as f:
    impute_stats = pickle.load(f)

# Fill missing values using the computed dataset median fallback
tabular = tabular.fillna(impute_stats.get("median_X", 0.0))
tabular.to_csv(DATA_DIR / "tabular_features.csv", index=False)

print(f"Cleaned tabular features saved. Remaining NaNs: {tabular.isna().sum().sum()}")

Cleaned tabular features saved. Remaining NaNs: 0


In [11]:
from pathlib import Path
import torch

# --- File Paths ---
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"
GRAPHS_PATH = DATA_DIR / "graphs.pt"

# --- Load and Inspect Graph Data ---
print(f"Loading {GRAPHS_PATH}...")
graphs = torch.load(GRAPHS_PATH)

print(f"\nTotal graphs loaded: {len(graphs)}")

# Grab the first sample
sample_graph = graphs[0]

print("\n--- Sample Graph Inspection ---")
print(sample_graph)

# Node Features
if hasattr(sample_graph, "x") and sample_graph.x is not None:
    print(
        f"\nNode Feature Matrix (x) Shape: {sample_graph.x.shape}"
    )  # [num_nodes, num_node_features]
    print(f"Sample Node Feature (Node 0): {sample_graph.x[0]}")
else:
    print("\nNode Features (x): Not found")

# Atomic Numbers (if stored separately)
if hasattr(sample_graph, "z") and sample_graph.z is not None:
    print(f"Atomic Numbers (z) Shape: {sample_graph.z.shape}")
    print(f"Sample Atomic Numbers (first 5): {sample_graph.z[:5]}")

# Edge Indices
if hasattr(sample_graph, "edge_index") and sample_graph.edge_index is not None:
    print(
        f"\nEdge Index Shape: {sample_graph.edge_index.shape}"
    )  # [2, num_edges]

# Edge Attributes
if hasattr(sample_graph, "edge_attr") and sample_graph.edge_attr is not None:
    print(
        f"Edge Feature Matrix (edge_attr) Shape: {sample_graph.edge_attr.shape}"
    )  # [num_edges, num_edge_features]
    print(f"Sample Edge Feature (Edge 0): {sample_graph.edge_attr[0]}")
else:
    print("Edge Features (edge_attr): Not found")

# Targets
if hasattr(sample_graph, "y") and sample_graph.y is not None:
    print(f"\nTarget Value (y): {sample_graph.y}")

Loading c:\projects\Materialmind\data\graphs.pt...


C:\Users\88kos\AppData\Local\Temp\ipykernel_3792\4074812225.py:11: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  graphs = torch.load(GRAPHS_PATH)
c:\projects\Materialmind\ve


Total graphs loaded: 10000

--- Sample Graph Inspection ---
Data(x=[12, 6], edge_index=[2, 144], edge_attr=[144, 2], y=[1, 1], material_id='mp-1524357')

Node Feature Matrix (x) Shape: torch.Size([12, 6])
Sample Node Feature (Node 0): tensor([83.0000,  2.0200,  1.6000,  7.2855, 15.0000,  8.0000])

Edge Index Shape: torch.Size([2, 144])
Edge Feature Matrix (edge_attr) Shape: torch.Size([144, 2])
Sample Edge Feature (Edge 0): tensor([2.3862, 1.0847])

Target Value (y): tensor([[1.8539]])
